# DHO1000 Rigol Waveform Examples

**awiamo**

**March 2026**

This notebook demonstrates extracting signals from `.wfm` and `.bin` files created by Rigol DHO800/DHO900/DHO1000 series oscilloscopes.

The DHO series supports two waveform export formats:

* **`.bin`** — Official binary format documented in the DHO1000 User Guide §19.2.4. Contains calibrated float32 voltage samples for each enabled channel.
* **`.wfm`** — Proprietary format containing zlib-compressed metadata blocks followed by raw uint16 ADC samples (reverse-engineered).

*If RigolWFM is not installed, uncomment the following cell (i.e., delete the #) and run (shift-enter)*

In [ ]:
#!pip install --user RigolWFM

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

try:
    import RigolWFM.wfm as rigol
except ModuleNotFoundError:
    print('RigolWFM not installed. To install, uncomment and run the cell above.')
    print('Once installation is successful, rerun this cell again.')

repo = "../wfm/"

A list of Rigol scopes in the DHO1000 family is:

In [ ]:
print(rigol.DHO1000_scopes)

## DHO1047 — Single-channel `.wfm` capture

### Look at a screen shot

This capture was taken on a DHO1074 oscilloscope with CH1 active, measuring a 3.3 V logic signal.

<img src="https://github.com/scottprahl/RigolWFM/raw/main/wfm/DHO1074.png" width="60%">

### Import the `.wfm` data

In [ ]:
filename = "DHO1047.wfm"
w = rigol.Wfm.from_file(repo + filename, 'DHO')

### Textual description of the waveform

In [ ]:
description = w.describe()
print(description)

### Plot the waveform

In [ ]:
ch = w.channels[0]

plt.plot(ch.times * 1e6, ch.volts, color='green')
plt.title("%s  CH%d  %.0f Sa/s" % (filename, ch.channel_number, 1 / ch.seconds_per_point))
plt.xlabel("Time (µs)")
plt.ylabel("Volts (V)")
plt.grid(True)

plt.show()

The `w.plot()` convenience method draws all enabled channels automatically:

In [ ]:
w.plot()
plt.show()

## DHO1074 — Four-channel `.bin` capture

### Look at a screen shot

This capture was taken on a DHO1074 oscilloscope with all four channels enabled.
The `.bin` format is documented in the DHO1000 User Guide, Section 19.2.4.
It stores calibrated float32 voltage values directly — no additional scaling is required.

<img src="https://github.com/scottprahl/RigolWFM/raw/main/wfm/DHO1074.png" width="60%">

### Import the `.bin` data

In [ ]:
filename = "DHO1074.bin"
w = rigol.Wfm.from_file(repo + filename, 'DHO')

### Textual description of the waveform

In [ ]:
description = w.describe()
print(description)

### Plot all four channels

The `.bin` format stores each channel independently. All four are plotted below.

In [ ]:
colors = ['green', 'red', 'blue', 'orange']
active = [ch for ch in w.channels if ch.volts is not None]
n = len(active)

fig, axes = plt.subplots(n, 1, sharex=True, figsize=(10, 2.5 * n))
if n == 1:
    axes = [axes]

for ax, ch, color in zip(axes, active, colors):
    ax.plot(ch.times * 1e6, ch.volts, color=color)
    ax.set_ylabel("Volts (V)")
    ax.set_title("CH%d  %.2f V/div" % (ch.channel_number, ch.volt_per_division))
    ax.grid(True)

axes[-1].set_xlabel("Time (µs)")
fig.suptitle(filename, y=1.01)
plt.tight_layout()
plt.show()

The `w.plot()` convenience method draws the same result in a single call:

In [ ]:
w.plot()
plt.show()

### Select a single channel from a multi-channel `.bin` file

Pass the `selected` argument to `Wfm.from_file()` to load only specific channels.
This avoids allocating memory for channels you do not need.

In [ ]:
filename = "DHO1074.bin"
w1 = rigol.Wfm.from_file(repo + filename, 'DHO', selected='1')

ch = w1.channels[0]
plt.title("CH%d  %.2f V/div  %.2f Voff  (%s)" % (
    ch.channel_number, ch.volt_per_division, ch.volt_offset, filename))
plt.plot(ch.times * 1e6, ch.volts, color='green', label='CH1')
plt.xlabel("Time (µs)")
plt.ylabel("Volts (V)")
plt.legend(loc='upper right')
plt.grid(True)
plt.show()

## Format comparison: `.wfm` vs `.bin`

DHO1047.wfm and DHO1074.bin are captures from the same oscilloscope model (DHO1074)
but from different measurement sessions, so a direct sample-by-sample comparison is
not meaningful here.  The table below summarises the key differences between the two formats.

| Property | `.bin` | `.wfm` |
|---|---|---|
| Documented | Yes (User Guide §19.2.4) | No (reverse-engineered) |
| Sample type | float32 (calibrated V) | uint16 (raw ADC) |
| Channels | All enabled, separate | CH1 only in tested files |
| Metadata | In waveform headers | zlib-compressed blocks |
| x_origin sign | Positive (negate for t0) | Negative (use directly) |

In [ ]:
wfm = rigol.Wfm.from_file(repo + "DHO1047.wfm", 'DHO')
wfm_ch = wfm.channels[0]

print("DHO1047.wfm")
print("  Points     : %d" % len(wfm_ch.times))
print("  Delta t    : %.3f µs" % (wfm_ch.seconds_per_point * 1e6))
print("  Time span  : %.3f ms" % ((wfm_ch.times[-1] - wfm_ch.times[0]) * 1e3))
print("  Vmin / Vmax: %.3f V / %.3f V" % (wfm_ch.volts.min(), wfm_ch.volts.max()))
print()

bin_ = rigol.Wfm.from_file(repo + "DHO1074.bin", 'DHO', selected='1')
bin_ch = bin_.channels[0]

print("DHO1074.bin  (CH1 only)")
print("  Points     : %d" % len(bin_ch.times))
print("  Delta t    : %.3f µs" % (bin_ch.seconds_per_point * 1e6))
print("  Time span  : %.3f ms" % ((bin_ch.times[-1] - bin_ch.times[0]) * 1e3))
print("  Vmin / Vmax: %.3f V / %.3f V" % (bin_ch.volts.min(), bin_ch.volts.max()))